# Phase 2 — Rerank (K1 features + K2 LightGBM LambdaMART)

Trains K2 on the train split (serve-identical query+fusion incl. dense-text), saves the model to Drive, and reports reranked vs fusion-only nDCG@20 on dev. Spec: `51_K2_lgbm_lambdamart.md`, `50_K1_*`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
_hf=(lambda n:(userdata.get(n) if True else None))
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
# pylate + peft: K2 trains on the SAME spine the blind submission serves (incl. the LoRA-ColBERT channel)
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate peft
!pip -q uninstall -y torchao   # transformers wants torchao>0.16 but Colab ships 0.10 and its check RAISES; we don't use it
import sys; sys.path.insert(0,'.')

In [ ]:
# OOM hygiene — MUST run before any import that pulls in JAX/torch (RESTART to apply mid-session).
# Root cause of the GPU OOMs: JAX preallocates 75% of VRAM on init, leaving PyTorch ~24% -> OOM.
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'   # JAX: allocate on demand (THE fix)
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
print('OOM-hygiene env set — RESTART if torch/jax were already imported')

## 3. Config

In [ ]:
TOPK=500; TRAIN_SESSIONS=7000; DEV_SESSIONS=1000
NEG_CAP=150            # random-sampled negatives/group (0 = all); preserves score dist
EARLY_STOPPING=50      # session-disjoint val + early stop on val ndcg@20 (0 = off)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '   # BGE asymmetric (R4 §4.3)
# kept channels after the P0 unique-recall ablation (dropped cknn_meta/lyrics):
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
# Frozen cross-encoder used ONLY as a K2 input FEATURE (ce_score; see the CE-stacking block below).
# bge-reranker-v2-m3 is loaded + scored here, NEVER trained — K2 (LGBM) is the only model this notebook
# trains. (Fine-tuning the cross-encoder lives in phase2_ce_finetune.ipynb.) Heavy: cost ~= (train+dev
# turns) x CROSS_ENCODER_K frozen forwards (fp16); keep CROSS_ENCODER_K small on a T4/G4.
CE_MODEL='BAAI/bge-reranker-v2-m3'; CROSS_ENCODER_K=50; CE_MAX_DOC_TOKENS=480
ORG='talkpl-ai'
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'    # A1 enriched docs (newest); train==serve on enriched

# ── K2 anti-overfit regularization (plan §9.1: shallow trees + L1/L2 + feature/row subsample) ──
# K2 was under-regularized (min_child_samples=5, no L1/L2/subsample) and memorized train (the leaky
# val callback in K3b read ~0.25 on K2's own train fold vs ~0.15 on unseen dev). These tighten it.
K2_PARAMS={'learning_rate':0.05,    # lower LR + more trees + early stop generalizes better than 0.1
           'num_leaves':15,         # shallow trees
           'min_child_samples':30,  # ↑ from 5 — the main overfit knob (min rows per leaf)
           'reg_lambda':5.0,        # L2
           'reg_alpha':1.0,         # L1
           'feature_fraction':0.8,  # column subsample per tree
           'bagging_fraction':0.8,'bagging_freq':1}   # row subsample

# ── CE stacking (frozen, leak-free) ──
# Stack a FROZEN bge-reranker score as a K2 feature instead of chaining K3 after K2. Leak-free with
# NO OOF: a frozen model never saw gold labels, so its (query,doc) score is a fixed function. K2 then
# LEARNS how to combine text-relevance with CF/popularity/history rather than letting the CE override.
CE_STACK=True

# ── ColBERT channel (LoRA fine-tune from phase2_colbert_finetune.ipynb) ──
# Train K2 on the SAME 8-channel spine the blind submission serves: base channels + the focused-query
# ColBERT PLAID channel. If K2 is trained WITHOUT colbert, the loaded model's feature set lacks
# rank_inv__colbert while the serve-side FeatureBuilder includes it -> the rerank() train==serve guard
# hard-fails. Requires the checkpoint at COLBERT_OUT_DIR (run phase2_colbert_finetune first).
# COLBERT_D_LEN / EXPANSION_FIRST MUST match phase2_colbert_finetune (the checkpoint was trained+indexed
# at these) AND phase3_blindA — all three share the PLAID index by signature.
USE_COLBERT=True
COLBERT_OUT_DIR=f'{OUT}/colbert/music-colbert-v1'
COLBERT_Q_LEN=96; COLBERT_D_LEN=512; COLBERT_BSIZE=128; COLBERT_EXPANSION_FIRST=True
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_BASE='music-colbert-v1-d%d'%COLBERT_D_LEN

## 4. Load data (HF) + build channels (enriched + BGE prefix + kept set: bm25/dense/cknn_audio/cknn_attr/cf/same_artist/related_artist)

In [ ]:
import glob, os, pickle, pandas as pd
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

# enriched catalog (A1) so train==serve docs match the probe
meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB))
if _enr:
    _edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
    cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
    print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')
else:
    cat=Catalog(meta_rows); USE_ENRICHED=False; print('WARNING: no enriched parquet -> RAW docs (run A1)')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names)
CKNN_MODS={lab:mod for lab,mod in CONTENT_MODALITIES.items() if mod in _avail}
te={lab:TrackEmbeddings(tre.select_columns(['track_id',mod]),modalities=[mod]) for lab,mod in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
conv_tr=Conversations(dsd['train'].select(range(TRAIN_SESSIONS))); conv_dv=Conversations(dsd['test'].select(range(DEV_SESSIONS)))

model=SentenceTransformer(DENSE_MODEL,device='cuda')
doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)

# related-artist cross-session co-occurrence, train-only (cached to Drive; shared with phase0)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
if os.path.exists(COOC_PKL):
    cooc=pickle.load(open(COOC_PKL,'rb')); print('loaded cooc',len(cooc),'artists')
else:
    cooc=build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat)); pickle.dump(cooc,open(COOC_PKL,'wb')); print('built cooc',len(cooc),'artists')

# kept channel set (distinct labels per content-kNN modality so fusion/features don't collide)
cknn=[ContentKNNChannel(te[lab],mod,label=lab) for lab,mod in CKNN_MODS.items()]
chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
       CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
fusion=RRFFusion(chans,k=60); labels=[c.label for c in chans]
print('channels:',labels)

In [ ]:
# ── ColBERT PLAID channel (focused-query routed) — the 8th spine channel K2 must train on, so the
# saved K2's feature set (incl. rank_inv__colbert) matches what phase3_blindA_submission serves. ──
if USE_COLBERT:
    import os, hashlib
    from pylate import models
    from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
    from mcrs.retrieval.colbert_channel import colbert_doc_text

    assert _enr, 'USE_COLBERT requires an enriched parquet (run A1) — the ColBERT checkpoint was trained/indexed on enriched docs; RAW docs would be a train/serve skew'
    assert os.path.isdir(COLBERT_OUT_DIR), f'ColBERT checkpoint missing at {COLBERT_OUT_DIR} — run phase2_colbert_finetune first'
    ft=models.ColBERT(model_name_or_path=COLBERT_OUT_DIR, query_length=COLBERT_Q_LEN, document_length=COLBERT_D_LEN)
    doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=COLBERT_EXPANSION_FIRST)
    # checkpoint-keyed + doc-version-keyed index name: a retrain OR a new enriched parquet/D_LEN/expansion
    # order busts a stale index (querying new vectors against old doc vectors = garbage). Mirrors phase3.
    _ck_sig=hashlib.md5('|'.join(f'{f}:{os.stat(os.path.join(rt,f)).st_size}:{int(os.stat(os.path.join(rt,f)).st_mtime)}'
                                 for rt,_,fs in os.walk(COLBERT_OUT_DIR) for f in sorted(fs)).encode()).hexdigest()[:8]
    _doc_sig=hashlib.md5(f'{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}|d{COLBERT_D_LEN}|ef{COLBERT_EXPANSION_FIRST}'.encode()).hexdigest()[:8]
    retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, f'{FT_IDX_BASE}-{_ck_sig}-{_doc_sig}', doc_fn, batch_size=COLBERT_BSIZE)

    class PlaidColBERTChannel:
        """Fusion channel backed by the fine-tuned ColBERT PLAID index. query_key='colbert' so RRF /
        build_rerank_groups route it the FOCUSED query (per_channel_queries) while base channels keep the
        full query — identical to the blind submission spine (train==serve)."""
        label='colbert'; query_key='colbert'
        def __init__(self, model, retriever): self.model, self.retriever = model, retriever
        def batch_text_to_item_retrieval(self, queries, topk, batch_context=None, user_ids=None):
            if not queries: return []
            return colbert_retrieve(self.model, self.retriever, list(queries), topk, batch_size=COLBERT_BSIZE)

    qb_focused=QueryBuilder(recency_window=1)          # ColBERT focused query (last utterance + goal)
    colbert=PlaidColBERTChannel(ft, retr)
    chans=chans+[colbert]
    fusion=RRFFusion(chans, k=60)
    labels=[c.label for c in chans]
    PCQ={'colbert': qb_focused}                         # per-channel routing for build_rerank_groups + InferenceHarness
    print('channels (w/ colbert):', labels)
else:
    PCQ=None
    print('channels (no colbert):', labels)

## 5. Train K2 + save model to Drive

In [ ]:
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.rerank.train import build_rerank_groups
from mcrs.rerank.neural import NeuralReranker, build_ce_score_lookup, make_ce_feature_fn
from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
import numpy as np, os, pickle
qb=QueryBuilder()

# dense query<->candidate cosine as a relevance FEATURE (the magnitude K2 was missing -> it leaned
# on popularity). Queries batch-encoded once per turn into _qcache; per-candidate = one dot product.
_qcache={}
def precompute_qvecs(turns):
    texts=[DENSE_QUERY_PREFIX+qb.build(t).text for t in turns]
    mat=model.encode(texts, batch_size=256, normalize_embeddings=True)
    _qcache.update({(t.session_id,t.turn_number): mat[i] for i,t in enumerate(turns)})
def dense_cos(ctx, tid):
    j=cat.id_to_index.get(tid)
    if j is None: return 0.0
    v=_qcache.get((ctx.session_id,ctx.turn_number))
    if v is None:                                   # fallback (turn not pre-encoded)
        v=model.encode([DENSE_QUERY_PREFIX+qb.build(ctx).text], normalize_embeddings=True)[0]
        _qcache[(ctx.session_id,ctx.turn_number)]=v
    return float(v @ doc_mat[j])

tr=list(conv_tr.turns()); precompute_qvecs(tr)
print('train turns:',len(tr),'- building groups (runs fusion incl dense over train)...')
# per_channel_query_builders=PCQ routes ColBERT's focused query EXACTLY like InferenceHarness, so the
# K2 training pool == the serve pool for every channel (no full-vs-focused skew on the routed channel).
groups=build_rerank_groups(qb,fusion,tr,lambda t:conv_tr.gold(t.session_id,t.turn_number),topk=TOPK,
                           per_channel_query_builders=PCQ)

score_fns={'dense_cos': dense_cos}
if CE_STACK:
    # FROZEN cross-encoder score stacked as a K2 feature (leak-free, NO OOF — frozen model never saw
    # the gold labels, so its (query,doc) score is a fixed function; see neural.py docstring). Scores
    # the top CROSS_ENCODER_K of each turn's pool, min-max normalized WITHIN that turn's pool only.
    # Includes DEV pools so the SAME ce_score feature is populated at serve (train==serve). Cached to
    # Drive — this is the slow step (~TRAIN+DEV turns x CROSS_ENCODER_K frozen-CE forwards on GPU).
    _CE_PKL=f'{OUT}/ce_stack_lookup_frozen_k{CROSS_ENCODER_K}_tr{TRAIN_SESSIONS}_dv{DEV_SESSIONS}.pkl'
    _CE_PARTIAL=_CE_PKL+'.partial'   # incremental checkpoint so a VM kill on the long CE pass is resumable
    if os.path.exists(_CE_PKL):
        ce_lookup=pickle.load(open(_CE_PKL,'rb')); print(f'loaded cached ce_lookup ({len(ce_lookup)}) <- {_CE_PKL}')
    else:
        _frozen_ce=build_cross_encoder_score_fn(CE_MODEL,device='cuda',max_length=512,
                                                max_doc_tokens=CE_MAX_DOC_TOKENS,dtype='fp16')  # no adapter -> frozen
        _nr=NeuralReranker(cat,qb,_frozen_ce,cross_encoder_k=CROSS_ENCODER_K,enriched=USE_ENRICHED)
        _dv=list(conv_dv.turns())
        # route the dev pool the SAME way (ColBERT focused) so the dev ce_score feature == serve
        _pcq_dv={'colbert':[qb_focused.build(t).text for t in _dv]} if (USE_COLBERT and PCQ) else None
        _dv_pools=fusion.fuse([qb.build(t).text for t in _dv],TOPK,
                              batch_context=[{'history_tids':t.history_tids,'user_id':t.user_id} for t in _dv],
                              user_ids=[t.user_id for t in _dv], per_channel_queries=_pcq_dv)
        # One work list (train groups + dev), scored in CHUNKS with a tqdm ETA (show_progress) and
        # CHECKPOINTED to Drive after each chunk -> a preemption loses at most one chunk, and a re-run
        # RESUMES from the partial (skips done turns). Each turn normalizes within its OWN pool, so
        # chunked == one pass (no cross-turn dependency).
        _turns=[g[0] for g in groups]+_dv
        _pools=[g[1] for g in groups]+_dv_pools
        ce_lookup=pickle.load(open(_CE_PARTIAL,'rb')) if os.path.exists(_CE_PARTIAL) else {}
        _done={(s,t) for (s,t,_t) in ce_lookup}
        _todo=[(c,p) for c,p in zip(_turns,_pools) if (c.session_id,c.turn_number) not in _done]
        _CE_CHUNK=10000
        print(f'frozen-CE: {len(_done)} turns already cached, scoring {len(_todo)} more in chunks of {_CE_CHUNK} (resumable)...')
        for _i in range(0,len(_todo),_CE_CHUNK):
            _ch=_todo[_i:_i+_CE_CHUNK]
            ce_lookup.update(build_ce_score_lookup([c for c,_ in _ch],[p for _,p in _ch],_nr,
                                                   normalize=True, show_progress=True))
            pickle.dump(ce_lookup,open(_CE_PARTIAL,'wb'))   # checkpoint (survives a VM kill)
            print(f'  checkpoint -> {min(_i+_CE_CHUNK,len(_todo))}/{len(_todo)} turns scored ({len(ce_lookup)} entries)')
        pickle.dump(ce_lookup,open(_CE_PKL,'wb'))            # mark complete
        if os.path.exists(_CE_PARTIAL): os.remove(_CE_PARTIAL)
        print(f'built + cached ce_lookup ({len(ce_lookup)}) -> {_CE_PKL}')
    # default=-1.0: candidates outside the CE-scored top-K get this sentinel (same at train & serve)
    score_fns['ce_score']=make_ce_feature_fn(ce_lookup, default=-1.0)

fb=FeatureBuilder(cat, labels, score_fns=score_fns)
k2=LGBMReranker(fb,n_estimators=500,params=K2_PARAMS,neg_cap=NEG_CAP,
                early_stopping_rounds=EARLY_STOPPING,val_fraction=0.1).fit(groups)
print(f'train_groups={k2.n_train_groups_} val_groups={k2.n_val_groups_} best_iter={getattr(k2.model,"best_iteration_",None)}')
k2.save(f'{OUT}/k2_lgbm.txt'); print('saved model ->',f'{OUT}/k2_lgbm.txt')
print('feature importances:',sorted(zip(fb.feature_names,k2.model.feature_importances_),key=lambda x:-x[1])[:10])

## 6. Eval reranked vs fusion-only (dev) → save scores

In [ ]:
import json
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness, validate_submission
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official

def _eval_set(turns, tag):
    # one reusable dev eval: fusion-only vs +K2 nDCG@20 over `turns`. _qcache is keyed on
    # (session_id, turn_number); refill it for THIS turn set so dense_cos is batched and no stale
    # (e.g. train) query vector leaks across splits. PCQ mirrors the train routing (ColBERT focused).
    _qcache.clear(); precompute_qvecs(turns)
    g=[GoldRow(t.session_id,t.user_id,t.turn_number,conv_dv.gold(t.session_id,t.turn_number)) for t in turns]
    k=[(x.session_id,x.turn_number) for x in g]
    b=InferenceHarness(qb,fusion,TopKAssembler(cat),topk=TOPK,per_channel_query_builders=PCQ).run(turns)
    r=InferenceHarness(qb,fusion,TopKAssembler(cat),reranker=k2,topk=TOPK,per_channel_query_builders=PCQ).run(turns)
    validate_submission(r,catalog=cat,expected_keys=k)
    sb=score_official(b,g,len(cat)); sr=score_official(r,g,len(cat))
    print(f'[{tag}] {len(turns)} turns | fusion-only nDCG@20={round(sb["ndcg@20"],4)} | +K2={round(sr["ndcg@20"],4)}')
    return {'fusion':sb,'reranked':sr,'n_turns':len(turns)}

# GATE = dev FINAL-turn proxy: Blind-A scores only each session's trailing turn, so K2 must be gated on
# THAT population (one gold-bearing trailing turn/session), not the all-turns set (shallow-turn-heavy,
# ~8x more rows — a distribution the leaderboard never sees). gold_target_turns() is target_turns()
# restricted to turns carrying a gold (Blind-A-comparable + scorable; mirrors phase3 dev-confirm).
ft=list(conv_dv.gold_target_turns())
final_scores=_eval_set(ft, 'GATE final-turn (Blind-A proxy)')
# secondary diagnostic: all gold-bearing turns (NOT the leaderboard population — do not gate on this).
allt_scores=_eval_set(list(conv_dv.turns()), 'diag all-turns')

scores={'final_turn':final_scores,'all_turns':allt_scores}
json.dump(scores, open(f'{OUT}/phase2_scores.json','w'), indent=2)
print('GATE: +K2 final-turn nDCG@20 must clearly beat fusion-only.  saved ->',f'{OUT}/phase2_scores.json')

## 7. Next
Gate K2 on the **final-turn** nDCG@20 (the `GATE final-turn` line — Blind-A scores only each session's trailing turn); the all-turns number is a secondary diagnostic, not the leaderboard population. The absolute ceiling is bounded by fused recall@K — to push it, lift recall (A1 enrichment / R6 / stronger dense / ColBERT), then retrain K2.